# 03 · 一个空间网络怎样产生传播，又怎样改变活动？

**建立 SNN → 定义拟合目标 → 新仿真与 core 搜索 → 传播比较 → Z/M 慢变量**

本篇可单独运行，逐步计算 Fig4、Fig5 的模型面板及 Video 2。输入是 E1146 的患者观测和坐标；网络、事件和动画在本次运行中生成。

默认 40,000 个神经元、每候选 12 秒；每个 core 数只评估 2 次，CPU 运行需要数十分钟或更久。模型用的 FIT/CAL 观测来自较完整记录，独立于前两本的 800 秒短例。


In [ ]:
import _setup
from modeling import computational as snn, panel_analysis as panels
from modeling.experiments import search_core_counts
from analysis import from_events as analysis, spectral_fields as spectral, model_transition as transition
from plotting import computational as plot
from workflows.tutorial import model_inputs, TutorialOutput, save_model_analysis

config, patient = model_inputs()
out = TutorialOutput('snn')


## 模型设定 · 局部兴奋性怎样影响传播？

在 20×20 mm 平面放置 80% E、20% I 神经元，用有延迟的空间连接耦合 LIF 动力学。core 降低局部 E 阈值，并接受独立 OU/Poisson 输入；虚拟触点读取附近活动。

位置、阈值和连接参数在 [snn_tutorial.json](../config/snn_tutorial.json)；下面新建网络。


In [ ]:
substrate = snn.build_substrate(config)


## 拟合目标 · 什么叫“接近患者”？

分别比较 **rank 模式、局部/跨杆时差、通道参与率** 三项分布。FIT 建立目标，独立 CAL 校准尺度，平均得到 $J$；越低表示分布差异越小。优化不使用 TA/TB 类别标签。


In [ ]:
objective = snn.fit_objective(patient.fit, patient.cal, patient.names,
    n_fourier=config['n_fourier'], cal_draw_indices=patient.cal_draw_indices)


## Fig4A · 先运行一个基准配置

固定 core 位置，积分膜电位和突触动力学，再从新事件计算 $J$。图中同时查看局部阈值与网络活动。


In [ ]:
baseline = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=config['training_seed'])
snn.score_simulation(baseline, patient.observer, objective, config['burnin_ms'])
out.show(plot.substrate_panel(baseline, config['radius_mm']), 'fig4_A_model')
out.simulation('baseline_new_simulation', baseline)


## Fig4F · 这些活动在触点上是什么波形？

将新 spike 做空间加权读出并滤到 30–80 Hz，观察各触点的先后。这里是模型活动代理，单位不同于 SEEG 电压。


In [ ]:
out.show(plot.bandpassed_model_waveform(baseline), 'fig4_F_baseline_waveforms')


## Core 优化 · 改变哪些参数，才能减小差异？

分别设置 1–4 个 core，用相同预算搜索位置与向外 E→E 权重。每个候选都重新仿真并计算 $J$；默认固定连接方向，随后单独扫描；完整配置也可联合优化方向。


In [ ]:
searches = search_core_counts(substrate, patient.contacts, patient.observer, objective,
    out.path/'core_search', core_counts=(1, 2, 3, 4),
    budget=config['optimization_budget'], restarts=config['restarts'])
rows = [row for search in searches for row in search['rows']]


## Fig4B · 哪一类可观察差异仍然存在？

把候选结果拆成平均 rank、杆内顺序和参与率误差，帮助解释拟合。这些直观误差与用于优化的分布统计量 $J$ 分开。


In [ ]:
out.show(plot.raw_errors(rows), 'fig4_B_observable_errors')


## Fig4C · 增加 core 是否更容易拟合？

比较各 core 数随评估次数变化的最佳训练 $J$。默认小预算用于走通方法，判断 core 数仍需足够的搜索与重复。


In [ ]:
out.show(plot.optimization_curves(searches), 'fig4_C_core_search')


## Fig4D · 传播对连接方向有多敏感？

固定其他参数和噪声，改变 E→E 连接核的角度，重新构图、仿真和评分。零偏移使用上面的基准结果。


In [ ]:
angle_rows = snn.angle_response(config, config['centers_mm'], patient.contacts, patient.observer,
    objective, angles=[a for a in config['angle_values'] if a != 0])
angle_rows = sorted([dict(angle=0, **baseline['score'])] + angle_rows, key=lambda row: row['angle'])
out.show(plot.raw_errors(angle_rows, angle=True), 'fig4_D_direction_response')


## Fig4E · 低误差配置的 core 位于哪里？

取本次有限 $J$ 配置中最低的 20%，汇总其 core 位置。它描述训练搜索偏好的位置。


In [ ]:
out.show(plot.core_density(rows, config['radius_mm']), 'fig4_E_core_positions')


## Fig4G · 基准模型是否产生了患者的传播顺序？

从患者 FIT 提取两个模板，给基准模型的新事件分配类别，比较平均归一化 rank。这一步读出拟合后的模式。


In [ ]:
matching = panels.model_patient_comparison(baseline['primary_centroids_ms'], patient.fit,
    patient.names, patient.contact_contract, n_null=1000)
out.show(plot.model_rank_profiles(matching, patient.names), 'fig4_G_mean_rank')


## Fig4H · 换一半触点评分，匹配还在吗？

一半触点用于分配模板，另一半计算相关，再交换两半。这避免用同一组触点既选类别又评价匹配。


In [ ]:
out.show(plot.model_crossfit_matrix(matching), 'fig4_H_crossfit')


## Video 2 · 波形与空间传播怎样对应？

固定 core，再模拟一个向外 E→E 权重为 1.0 的工作点，与基准 1.25 比较。每个工作点取各模板类别最早的合格事件，按自身质心对齐，观察波形与活动场；图中同时标出 $J$。


In [ ]:
comparison_run = snn.simulate(substrate, config['centers_mm'], patient.contacts,
    outward=1.0, seed=config['training_seed'])
snn.score_simulation(comparison_run, patient.observer, objective, config['burnin_ms'])
patient_templates = analysis.cluster_events(panels.centroid_events(patient.fit, patient.names))
comparison, selection = panels.select_propagation_events([baseline, comparison_run],
    patient_templates['templates'], patient.names, config['burnin_ms'])
plot.selected_propagation_movie(comparison, selection, out.figures/'supplementary_video2_new.gif', config['radius_mm'])
out.simulation('outward_100', comparison_run)


## Fig4I 的单患者示例 · 换一次输入噪声会怎样？

保持基准配置，更换噪声种子后独立评分。它检查同一个模型对噪声的敏感性；论文患者队列需逐患者重复拟合。


In [ ]:
confirmation = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=928401)
snn.score_simulation(confirmation, patient.observer, objective, config['burnin_ms'])
out.show(plot.noise_confirmation(baseline, confirmation), 'fig4_I_noise_confirmation')
out.simulation('new_noise_confirmation', confirmation)


## Fig5A · 加入慢变量后，活动如何改变？

保持网络与输入设定，启用调节抑制效能的 Z 和适应电流 M。重新积分并记录突触电流代理，观察活动进入高发放率阶段的过程。


In [ ]:
zm = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=config['training_seed'],
    zm=config['zm'], record_current=True, duration_ms=18000.)
onset = transition.display_entry(zm)
entry = spectral.native_entry(zm['time_ms'], zm['rate_E'], dt_ms=config['parameters']['dt'])
out.show(plot.raw_current_waveforms(zm, onset=onset), 'fig5_A_waveforms')
out.simulation('zm_enabled', zm)


## Fig5B · 慢变量怎样随活动演化？

查看同次仿真的全体 E 平均 Z 与有效适应电流 $\eta M$，把慢变化与上面的波形对应起来。


In [ ]:
out.show(plot.slow_variables(zm, config['zm']['eta_m']), 'fig5_B_slow_variables')


## Fig5C · 活动如何在空间中展开？

在进入时刻前后固定取四个时点，查看各自 50 ms 的平均活动。若尚未进入，则按固定间隔展示。


In [ ]:
out.show(plot.activity_snapshots(zm, config['radius_mm'], onset=onset), 'fig5_C_spatial_activity')


## Fig5D · 把活动和慢变量放在一起

沿同一时间轴配对 Z、$\eta M$ 与 E 发放率，得到三维轨迹。它描述这次状态变化，单条轨迹并不证明存在吸引子。


In [ ]:
out.show(plot.state_trajectory(zm, config['zm']['eta_m']), 'fig5_D_state_trajectory')


## Fig5E · 活动变化对应怎样的能量场？

用患者间期模板固定投影几何。从新电流计算进入后 1 秒的 1–150 Hz 功率，相对基线转成 robust-z，再投影到这个固定平面。


In [ ]:
energy = spectral.energy_field_trajectory(zm['contact_current'], 2000., patient.contacts,
    patient_templates['templates'][0], patient_templates['support'][0], baseline_sec=(1.5, 3.), band=(30, 80))
onset_field = transition.onset_broadband_field(zm['contact_current'], 2000., onset)
out.show(plot.onset_energy_field(zm, onset_field, energy), 'fig5_E_model_onset_field')


## 补充 · 能量与间期场的对应怎样随时间改变？

另取 30–80 Hz 能量，连续计算与固定间期场的相关和投影。这样可同时观察能量大小与空间分布的变化。


In [ ]:
out.show(plot.energy_alignment(energy), 'supplementary_gamma_field_scores')


## 这条计算链回答到哪里？

我们从新网络得到波形、传播与慢变量下的活动，并用患者观测评价它们。几何是建模先验，高发放率本身不等于临床发作。

默认搜索采用 Sobol + 局部精修，区别于历史 GP 优化；小预算结果用于方法演示。完整 Fig4I、Fig5E/F 的临床比较还需相应原始队列。扩大预算、临床接口和精确读出定义见 [方法说明](../METHODS_AND_PANELS.md#notebook3fig4fig5及video2)。


In [ ]:
save_model_analysis(out, config, objective, matching, selection, onset, entry, onset_field, energy,
    angle_rows=angle_rows)
